# Flight dataframe cleaning
Cleaning flight data obtained from web-scraper + adding airport locations

## 1. Imports & Setup

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

## 2. Examining Flights Data

In [29]:
final_df = pd.read_csv("departures_2024_all.csv")
print(final_df.info())
print(final_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967209 entries, 0 to 3967208
Data columns (total 19 columns):
 #   Column                                    Dtype  
---  ------                                    -----  
 0   Carrier Code                              object 
 1   Date (MM/DD/YYYY)                         object 
 2   Flight Number                             float64
 3   Tail Number                               object 
 4   Destination Airport                       object 
 5   Scheduled departure time                  object 
 6   Actual departure time                     object 
 7   Scheduled elapsed time (Minutes)          float64
 8   Actual elapsed time (Minutes)             float64
 9   Departure delay (Minutes)                 float64
 10  Wheels-off time                           object 
 11  Taxi-Out time (Minutes)                   float64
 12  Delay Carrier (Minutes)                   float64
 13  Delay Weather (Minutes)                   float64
 14  De

## 3. Cleaning Flights Data

In [68]:
flights_df["Carrier Code"].unique()

array(['DL', 'UA', 'AA', 'WN'], dtype=object)

In [75]:
flights_df = final_df.copy()

# Drop irrelevant columns
flights_df = flights_df.drop(columns=["Flight Number", "Tail Number", "Wheels-off time",
                                       "Delay Carrier (Minutes)", "Delay National Aviation System (Minutes)",
                                       "Delay Security (Minutes)", "Delay Late Aircraft Arrival (Minutes)", "airline"])

# Extract airport code from airport column
flights_df["airport_code"] = flights_df["airport"].str.extract(r'\((\w+)\)$')
flights_df = flights_df.drop(columns=["airport"])

# Parse date and extract useful features
flights_df["Date (MM/DD/YYYY)"] = pd.to_datetime(flights_df["Date (MM/DD/YYYY)"])
flights_df["month"] = flights_df["Date (MM/DD/YYYY)"].dt.month
flights_df["day_of_week"] = flights_df["Date (MM/DD/YYYY)"].dt.dayofweek
flights_df = flights_df.drop(columns=["Date (MM/DD/YYYY)"])

# Extract hour from scheduled departure time
flights_df["scheduled_hour"] = pd.to_datetime(flights_df["Scheduled departure time"], format="%H:%M").dt.hour
flights_df = flights_df.drop(columns=["Scheduled departure time", "Actual departure time"])

# Fill NaN delay values with 0
flights_df["Delay Weather (Minutes)"] = flights_df["Delay Weather (Minutes)"].fillna(0)
flights_df["Departure delay (Minutes)"] = flights_df["Departure delay (Minutes)"].fillna(0)

# Create target variable
flights_df["weather_delayed"] = (flights_df["Delay Weather (Minutes)"] > 0).astype(int)
flights_df = flights_df.drop(columns=["Delay Weather (Minutes)"])

# Drop remaining rows with NaN
flights_df = flights_df.dropna()

# Rename airport code columns
flights_df = flights_df.rename(columns={"airport_code": "origin_code", "Destination Airport": "destination_code"})

print(flights_df.info())
flights_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 3966862 entries, 0 to 3967207
Data columns (total 11 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Carrier Code                      object 
 1   destination_code                  object 
 2   Scheduled elapsed time (Minutes)  float64
 3   Actual elapsed time (Minutes)     float64
 4   Departure delay (Minutes)         float64
 5   Taxi-Out time (Minutes)           float64
 6   origin_code                       object 
 7   month                             float64
 8   day_of_week                       float64
 9   scheduled_hour                    float64
 10  weather_delayed                   int64  
dtypes: float64(7), int64(1), object(3)
memory usage: 363.2+ MB
None


,Carrier Code,destination_code,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),origin_code,month,day_of_week,scheduled_hour,weather_delayed
0,DL,DTW,110.0,92.0,12.0,12.0,EWR,1.0,0.0,13.0,0
1,DL,SLC,325.0,291.0,-5.0,23.0,EWR,1.0,0.0,7.0,0
2,DL,MSP,184.0,152.0,-10.0,13.0,EWR,1.0,0.0,12.0,0
3,DL,MSP,194.0,161.0,4.0,20.0,EWR,1.0,0.0,18.0,0
4,DL,SLC,330.0,278.0,18.0,17.0,EWR,1.0,0.0,18.0,0


## 4. Obtaining Airport Location Data

In [73]:
airport_coords_df = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv")
airport_coords_df = airport_coords_df[airport_coords_df["iata_code"].notna()]
airport_coords_df = airport_coords_df[airport_coords_df["iso_country"]=="US"]
airport_coords_df = airport_coords_df[["iata_code", "latitude_deg", "longitude_deg", "name"]]
airport_coords_df

,iata_code,latitude_deg,longitude_deg,name
410,OCA,25.325399,-80.274803,Ocean Reef Club Airport
632,CSE,38.851918,-106.928341,Crested Butte Airpark
888,CUS,31.823898,-107.629924,Columbus Airport
985,JCY,30.251801,-98.622498,LBJ Ranch Airport
1284,WLR,55.601299,-131.636993,Loring Seaplane Base
...,...,...,...,...
83844,CZP,55.966301,-133.796997,Cape Pole Seaplane Base
83845,KBW,56.295601,-158.401001,Chignik Bay Seaplane Base
83849,KBC,66.274002,-145.824005,Birch Creek Airport
83851,CZC,61.943713,-145.299398,Copper Center 2 Airport


## 5. Merging Location and Flight Data

In [82]:
# Origin coordinates
flights_df = flights_df.merge(airport_coords_df[["iata_code", "latitude_deg", "longitude_deg"]],
                               left_on="origin_code", right_on="iata_code", how="left")
flights_df = flights_df.drop(columns=["iata_code"])
flights_df = flights_df.rename(columns={"latitude_deg": "origin_lat", "longitude_deg": "origin_lon"})

# Destination coordinates
flights_df = flights_df.merge(airport_coords_df[["iata_code", "latitude_deg", "longitude_deg"]],
                               left_on="destination_code", right_on="iata_code", how="left")
flights_df = flights_df.drop(columns=["iata_code"])
flights_df = flights_df.rename(columns={"latitude_deg": "dest_lat", "longitude_deg": "dest_lon"})

print(flights_df.info())
flights_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3966862 entries, 0 to 3966861
Data columns (total 17 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Carrier Code                      object 
 1   destination_code                  object 
 2   Scheduled elapsed time (Minutes)  float64
 3   Actual elapsed time (Minutes)     float64
 4   Departure delay (Minutes)         float64
 5   Taxi-Out time (Minutes)           float64
 6   origin_code                       object 
 7   month                             float64
 8   day_of_week                       float64
 9   scheduled_hour                    float64
 10  weather_delayed                   int64  
 11  origin_lat                        float64
 12  origin_lon                        float64
 13  origin_lat                        float64
 14  origin_lon                        float64
 15  dest_lat                          float64
 16  dest_lon                          fl

,Carrier Code,destination_code,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),origin_code,month,day_of_week,scheduled_hour,weather_delayed,origin_lat,origin_lon,origin_lat,origin_lon,dest_lat,dest_lon
0,DL,DTW,110.0,92.0,12.0,12.0,EWR,1.0,0.0,13.0,0,40.6894,-74.170545,40.6894,-74.170545,42.213770,-83.353786
1,DL,SLC,325.0,291.0,-5.0,23.0,EWR,1.0,0.0,7.0,0,40.6894,-74.170545,40.6894,-74.170545,40.788860,-111.979866
2,DL,MSP,184.0,152.0,-10.0,13.0,EWR,1.0,0.0,12.0,0,40.6894,-74.170545,40.6894,-74.170545,44.880081,-93.221741
3,DL,MSP,194.0,161.0,4.0,20.0,EWR,1.0,0.0,18.0,0,40.6894,-74.170545,40.6894,-74.170545,44.880081,-93.221741
4,DL,SLC,330.0,278.0,18.0,17.0,EWR,1.0,0.0,18.0,0,40.6894,-74.170545,40.6894,-74.170545,40.788860,-111.979866


In [84]:
flights_df.to_csv("flights.csv", index=False)